In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("train.csv")

In [2]:
#پرکردن خانه های خالی از جنس استرینگ

df["PoolQC"] = df["PoolQC"].fillna("No Pool")
df["MiscFeature"] = df["MiscFeature"].fillna("None")
df["Alley"] = df["Alley"].fillna("No alley access")
df["Fence"] = df["Fence"].fillna("No Fence")
df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["FireplaceQu"] = df["FireplaceQu"].fillna("No Fireplace")
df["GarageQual"] = df["GarageQual"].fillna("No Garage")
df["GarageFinish"] = df["GarageFinish"].fillna("No Garage")
df["GarageType"] = df["GarageType"].fillna("No Garage")
df["GarageCond"] = df["GarageCond"].fillna("No Garage")
df["BsmtFinType2"] = df["BsmtFinType2"].fillna("No Basement")
df["BsmtExposure"] = df["BsmtExposure"].fillna("No Basement")
df["BsmtCond"] = df["BsmtCond"].fillna("No Basement")
df["BsmtQual"] = df["BsmtQual"].fillna("No Basement")
df["BsmtFinType1"] = df["BsmtFinType1"].fillna("No Basement")
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

#پرکردن خانه های خالی از جنس عدد
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

####################################
#مهندسی فیچر

df["HouseAge"] = 2010 - df["YearBuilt"]

df["QualLivArea"] = df["OverallQual"] * df["GrLivArea"]

df["LotArea"] = np.log1p(df["LotArea"])

df["BsmtFinishedRatio"] = np.where(
    df["TotalBsmtSF"] > 0,#چون اگر TotalBsmtSF صفر باشد، از تقسیم بر صفر جلوگیری می‌شود.
    df["BsmtFinSF1"] / df["TotalBsmtSF"], 0)

df["QualCond"] = df["OverallQual"] * df["OverallCond"]

df["TotalBath"] = (
    df["FullBath"]
    + 0.5 * df["HalfBath"]
    + df["BsmtFullBath"]
    + 0.5 * df["BsmtHalfBath"])

df["TotalSF"] = df["TotalBsmtSF"] + df["GrLivArea"]

df["RemodAge"] = 2010 - df["YearRemodAdd"]

df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)

df["HasFence"] = (df["Fence"] != "No Fence").astype(int)

df["QualFireplace"] = df["OverallQual"] * df["Fireplaces"]

#######################################################

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


y = np.log1p(y)

#چون فقط امتیاز میانگین برامون مهمه model.score(X_test, y_test) رو حساب نمیخوایم کنیم فعلا
#و برای محاسبه امیتاز میانگین هم نیاز به اسپلیت کردن دیتا نداریم 
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

In [3]:
#فعلاً همه پارامترها را روی مقدار پیش‌فرض بگذار و فقط n_estimators را تغییر بده.

#n_estimators_list = [100, 200, 300, 500, 700, 1000]
#for n in n_estimators_list:
#    model = RandomForestRegressor(
#        n_estimators=n,
#        random_state=42
#    )
    
    #امتیازمیانگین که باید داخل لوپ باشه
#    from sklearn.model_selection import cross_val_score# این داخل حلفه هست چون هربار که ان عوض میشه باید یدور امتیاز میانگین رو براش حساب و چاپ کنیم
#    scores = cross_val_score(model, X, y, cv=5)

#    print(
#        f"n_estimators={n} | Mean CV = {scores.mean():.6f}"
#    )

In [4]:
#آخرین امتیاز میانگین مدلمون بعد از مهندسی فیچر 0.8834220102732209 بود 
# نتیجه بررسی n_estimators نشان داد که با افزایش تعداد درخت‌ها، عملکرد مدل بهبود پیدا می‌کند.
#بین اینا n_estimators=1000 | Mean CV = 0.884807 از امتیاز میانگین بیشتر شد 
# اختلاف عملکرد بین 500 و 1000 بسیار کم است (حدود 0.00007)، 
#اگر هدف کسب بهترین امتیاز ممکن در کگل بود مقدار 1000 انتخاب میکردم اما چون الان هدفم سریعتر بودن محاسباته 500 انتخاب میکنم.
# از این مرحله به بعد n_estimators را روی 500 ثابت نگه می‌داریم
# و فقط هایپرپارامتر بعدی را تنظیم می‌کنیم

In [6]:
#max_depth
#این پارامتر مشخص می‌کند هر درخت حداکثر چند لایه (عمق) داشته باشد.
#عمق کم → مدل ساده‌تر، احتمال Underfitting
#عمق زیاد → مدل پیچیده‌تر، احتمال Overfitting
#max_depth_list = [5, 10, 15, 20, 30, None] #یعنی هیچ محدودیتی برای عمق درخت نگذار درخت آن‌قدر رشد می‌کند تا دیگر نتواند تقسیم جدیدی انجام دهد یا به محدودیت‌های دیگر مدل برسد.

#for depth in max_depth_list:

#    model = RandomForestRegressor(
 #       n_estimators=500,
 #       max_depth=depth,
 #       random_state=42
  #  )
    
  #  scores = cross_val_score(model, X, y, cv=5)

 #   print(
 #       f"max_depth={depth} | Mean CV = {scores.mean():.6f}"
  #  )

max_depth=5 | Mean CV = 0.856833
max_depth=10 | Mean CV = 0.883278
max_depth=15 | Mean CV = 0.884588
max_depth=20 | Mean CV = 0.884560
max_depth=30 | Mean CV = 0.884733
max_depth=None | Mean CV = 0.884733


In [ ]:
#max_depth=5 | Mean CV = 0.856833
#max_depth=10 | Mean CV = 0.883278
#max_depth=15 | Mean CV = 0.884588
#max_depth=20 | Mean CV = 0.884560
#max_depth=30 | Mean CV = 0.884733
#max_depth=None | Mean CV = 0.884733

#max_depth=15 | Mean CV = 0.884588 رو انتخاب میکنم 

In [7]:
#min_samples_split

#min_samples_split_values = [2, 5, 10, 20]

#for value in min_samples_split_values:
   # model = RandomForestRegressor(
   #     n_estimators=500,      # بهترین مقداری که از مرحله قبل پیدا کردی
   #     max_depth=15,          # بهترین مقدار از تست فعلی
  #      min_samples_split=value,
   #     random_state=42,
  #      n_jobs=-1 #این واسه اینه که از تمام cpu های لپتاپم استفاده کنم تا سریعتر بشه
 #   )
    
  #  scores = cross_val_score(
   #     model,
   #     X,
   #     y,
   #     cv=5,
   #     scoring="r2"
 #   )
    
  #  print(f"min_samples_split={value} | Mean CV = {scores.mean():.6f}")

min_samples_split=2 | Mean CV = 0.884588
min_samples_split=5 | Mean CV = 0.884413
min_samples_split=10 | Mean CV = 0.883101
min_samples_split=20 | Mean CV = 0.877184


In [ ]:
#min_samples_split=2 | Mean CV = 0.884588
#in_samples_split=5 | Mean CV = 0.884413
#min_samples_split=10 | Mean CV = 0.883101
#min_samples_split=20 | Mean CV = 0.877184

#بهترین مقدار همان مقدار پیش‌فرض Random Forest یعنی: min_samples_split=2 یعنی محدود کردن رشد درخت برای این دیتاست مفید نبوده.

In [8]:
#min_samples_leaf این پارامتر معمولاً در Random Forest اثر بیشتری از min_samples_split دارد.
#حداقل تعداد نمونه‌هایی که باید در هر برگ نهایی درخت باقی بماند
#یعنی وقتی درخت تصمیم می‌گیرد یک شاخه را ادامه دهد، آخرین قسمت‌های درخت (جایی که دیگر تقسیم نمی‌شود) باید حداقل این تعداد داده داشته باشند.

#min_samples_leaf_values = [1, 2, 4, 8, 10]

#for value in min_samples_leaf_values:
    # model = RandomForestRegressor(
      #  n_estimators=500,
      #  max_depth=15,
     #   min_samples_split=2,
      #  min_samples_leaf=value,
      #  random_state=42,
     #   n_jobs=-1
    #)
    
  #  scores = cross_val_score(
    #    model,
    #    X,
     #   y,
      #  cv=5,
       # scoring="r2"
  #  )
    
   # print(f"min_samples_leaf={value} | Mean CV = {scores.mean():.6f}")

min_samples_leaf=1 | Mean CV = 0.884588
min_samples_leaf=2 | Mean CV = 0.883552
min_samples_leaf=4 | Mean CV = 0.880050
min_samples_leaf=8 | Mean CV = 0.872744
min_samples_leaf=10 | Mean CV = 0.869188


In [ ]:
#min_samples_leaf=1 | Mean CV = 0.884588
#min_samples_leaf=2 | Mean CV = 0.883552
#min_samples_leaf=4 | Mean CV = 0.880050
#min_samples_leaf=8 | Mean CV = 0.872744
#min_samples_leaf=10 | Mean CV = 0.869188

#یعنی برای این دیتاست، محدود کردن برگ‌ها باعث underfitting شده. 
# min_samples_leaf=1 رو نگه میداریم پس همون دیفالت ینی

In [9]:
#max_features این پارامتر می‌گوید در هر اسپلیت درخت، چند تا از ویژگی‌ها بررسی شوند.
#در رندوم فورست ما چندین دسیژن تری میسازیم وقتی یک درخت میخواد یه اسپلیت انجام بده باید تصمیم بگیره از کدوم ویژگی ها استفاده کنه
#اگر max_features=1.0 ، هر درخت در هر اسپلیت تمام ویژگی‌ها را می‌بیند.
#اگر max_features="sqrt" تعداد ویژگی‌هایی که در هر اسپلیت بررسی می‌شود: جذر تعداد فیچرهاست
#اگر max_features="log2" ینی در هر اسپلیت لگاریتم بر پایه 2 تعداد ویژگی ها بررسی میشود از جذرویژگیها محدودکننده‌تر است.

#مزیتش تنوع بیشتر درخت ها
#max_features_values = ["sqrt", "log2", 0.5, 0.8, 1.0]

#for value in max_features_values:
 #   model = RandomForestRegressor(
 #       n_estimators=500,
 #       max_depth=15,
 #       min_samples_split=2,
#        min_samples_leaf=1,
#        max_features=value,
#        random_state=42,
 #       n_jobs=-1
 #   )
    
  #  scores = cross_val_score(
 #       model,
 #       X,
  #      y,
  #      cv=5,
  #      scoring="r2"
  #  )
  
 #   print(f"max_features={value} | Mean CV = {scores.mean():.6f}")

max_features=sqrt | Mean CV = 0.885174
max_features=log2 | Mean CV = 0.868400
max_features=0.5 | Mean CV = 0.888667
max_features=0.8 | Mean CV = 0.886007
max_features=1.0 | Mean CV = 0.884588


In [ ]:
#max_features=sqrt | Mean CV = 0.885174
#max_features=log2 | Mean CV = 0.868400
#max_features=0.5 | Mean CV = 0.888667
#max_features=0.8 | Mean CV = 0.886007
#max_features=1.0 | Mean CV = 0.884588

##max_features=0.5 | Mean CV = 0.888667 رو انتخاب میکنیم 

In [11]:
# قبل از تغییر پارامترها امتیاز میانگین 0.8834220102732209  بود و در آخر به  0.888667 رسید که افزایش قابل توجهیه حدود +0.005 بهبود داشتیم
#برای پروژه Kaggle این افزایش کوچک نیست، چون مدل‌های درختی بعد از یک نقطه با تغییر پارامترها معمولاً چند هزارم بهتر می‌شوند.
#حالا دو کار منطقی داریم:
#1. یک تست کوچک برای n_estimators دوباره با این تنظیمات انجام دهیم (چون قبلاً با تنظیمات قدیمی تست کردیم).
#2. اگر بهتر نشد، مدل نهایی رو ترین کنیم و با دیتای مسابقه کار کنیم


#n_estimators_values = [300, 500, 800]

#for value in n_estimators_values:
#    model = RandomForestRegressor(
 #       n_estimators=value,
 #       max_depth=15,
 #       min_samples_split=2,
 #       min_samples_leaf=1,
#       max_features=0.5,
 #       random_state=42,
 #       n_jobs=-1
 #   )
    
 #   scores = cross_val_score(
 #       model,
  #      X,
   #     y,
   #     cv=5,
   #     scoring="r2"
  #  )
    
   # print(f"n_estimators={value} | Mean CV = {scores.mean():.6f}")

In [ ]:
#n_estimators=300 | Mean CV = 0.888255
#n_estimators=500 | Mean CV = 0.888667
#n_estimators=800 | Mean CV = 0.888893

#n_estimators=800 | Mean CV = 0.888893 رو انتخاب میکنم

In [12]:
# در نهایت بهترین تنطیم ما اینا شد
RandomForestRegressor(
    n_estimators=800,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.5,
    random_state=42,
    n_jobs=-1
)

from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()
#و بهترین امتیاز Cross Validation: Mean CV = 0.888893

np.float64(0.8888926163174456)